<div style="text-align: right; font-weight: bold;">
東京工科大学コンピュータサイエンス学部<br>
福西広晃
</div>

----

# 感情分析の説明

- 感情分析とは、文章に含まれる**肯定的・否定的な感情や評価を自動的に判定するテキスト分析手法**である。

- 例えば、次のような文章を分析することができる。
  - 商品レビュー
  - 店舗の口コミ
  - SNSへの投稿
  - アンケートの自由記述

- 感情分析を行うことで、文章を書いた人が**どのような印象や評価を持っているか**を分析できる。

- 今回は、カフェの口コミデータに対して、**学習済みの自然言語処理モデル**を用いて感情分析を行う。

- 各口コミの文章をモデルに入力し、主に次のような感情に分類する。
  - **Positive**：肯定的な内容
  - **Negative**：否定的な内容

- 感情分析の結果について、次の内容を確認する。
  - Positive、Negativeと判定された**口コミの件数**
  - Positive、Negativeの**割合**
  - **星評価（1～5）と感情分析結果の関係**

- これにより、**数値として与えられた星評価と、口コミ文章から読み取れる感情との関係**を分析する。


----

# データの読み込み

**プロンプトの例**
「Google Driveの マイドライブ 内にある データ フォルダの中の テキスト フォルダに入っている 架空カフェ店口コミデータ.xlsx というExcelファイルを読み込んで、表示してください。」

# 感情分析の実行

**プロンプトの例**
「読み込んだ口コミデータに対して感情分析を実行してください」

# 感情分析結果の解析

**プロンプトの例**
「感情分析した口コミデータを分析して、結果がわかりやすくなるように集計やグラフを作成してください。」

----
----
----
----
----
----
----
----
----
----

# プログラム例

## データの読み込み

In [5]:
import pandas as pd
import os

# Excelファイルのパスを更新
#file_path = '/content/drive/MyDrive/東京成徳大学特別講義/2026年度/第02回/データ/テキスト/架空カフェ店口コミデータ.xlsx'
#file_path = '/content/drive/MyDrive/tsu_2026_github/データ/テキスト/架空カフェ店口コミデータ.xlsx'
file_path = '../データ/テキスト/架空カフェ店口コミデータ.xlsx'

# ファイルが存在するか確認
if not os.path.exists(file_path):
    print(f"エラー: 指定されたパスにファイルが見つかりません。パスを確認してください: {file_path}")
    print("Googleドライブの「マイドライブ」からの正確なフォルダ階層とファイル名をご確認ください。")
    df = pd.DataFrame() # 空のDataFrameを作成して続行
else:
    # ExcelファイルをDataFrameに読み込む
    try:
        df = pd.read_excel(file_path)
        print("データを読み込みました。")
        display(df.head())
    except Exception as e:
        print(f"エラー: Excelファイルの読み込み中に問題が発生しました。エラー内容: {e}")
        print("Google Driveの接続が不安定な可能性があります。もう一度実行してみてください。")
        df = pd.DataFrame() # 空のDataFrameを作成して続行

データを読み込みました。


,ID,評価(1-5),口コミ
0,1,4,店内は落ち着いた雰囲気で、コーヒーもとても美味しかったです。良い時間を過ごせました。
1,2,2,混雑していて席がなかなか空きませんでした。あまりおすすめできません。
2,3,1,混雑していて席がなかなか空きませんでした。次は別のお店を利用するかもしれません。
3,4,4,スタッフの対応が丁寧で気持ちよく過ごせました。友人にもおすすめしたいです。
4,5,5,アクセスが良く、また来たいと思えるカフェでした。良い時間を過ごせました。


## 感情分析の実行

In [11]:
import subprocess
import sys

print("必要なライブラリをインストール中...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib_fontja"])
print("インストール完了\n")

from sentence_transformers import SentenceTransformer, util
import numpy as np
import matplotlib_fontja

print("Sentence Transformer モデルを読み込み中...")

# 多言語対応のSentenceTransformerモデルを使用
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')

print("感情分析モデルを読み込みました。\n")

# 感情参照テキスト（日本語）
positive_refs = [
    "いい", "良い", "素晴らしい", "最高", "満足",
    "美味しい", "楽しい", "嬉しい", "気持ちいい", "おすすめ"
]
negative_refs = [
    "悪い", "最悪", "不満", "うるさい", "がっかり",
    "つまらない", "不快", "不愉快", "イライラ", "がっくり"
]

# 参照テキストをエンコード
positive_embedding = model.encode(positive_refs)
negative_embedding = model.encode(negative_refs)

# 参照ベクトルを平均化
positive_vec = positive_embedding.mean(axis=0)
negative_vec = negative_embedding.mean(axis=0)

print("口コミの感情分析を実行中...\n")

# 口コミに対して感情分析を実行
emotions = []
scores = []

for idx, text in enumerate(df["口コミ"], 1):
    try:
        # テキストをエンコード
        text_embedding = model.encode(text)
        
        # 肯定・否定感情との類似度を計算（コサイン類似度）
        positive_score = float(util.cos_sim(text_embedding, positive_vec).numpy().flatten()[0])
        negative_score = float(util.cos_sim(text_embedding, negative_vec).numpy().flatten()[0])
        
        # スコアをシグモイド関数で0-1の範囲に正規化
        def sigmoid(x):
            return 1 / (1 + np.exp(-x))
        
        # 差分をシグモイドで変換
        sentiment_diff = positive_score - negative_score
        normalized_score = sigmoid(sentiment_diff * 3)  # スケーリングファクター 3
        
        # 判定
        if normalized_score > 0.55:
            emotion = 'POSITIVE'
            score = normalized_score
        elif normalized_score < 0.45:
            emotion = 'NEGATIVE'
            score = 1 - normalized_score
        else:
            emotion = 'NEUTRAL'
            score = 0.5
        
        emotions.append(emotion)
        scores.append(score)
        
        print(f"{idx}. {text[:30]}... → {emotion} ({score:.2%})")
    
    except Exception as e:
        print(f"分析エラー（行{idx}）: {e}")
        emotions.append('NEUTRAL')
        scores.append(0.5)

print("\n感情分析が完了しました。\n")

# 結果をDataFrameに追加
df["感情"] = emotions
df["感情スコア"] = scores

display(df.head())


必要なライブラリをインストール中...
インストール完了

Sentence Transformer モデルを読み込み中...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2398.43it/s]


感情分析モデルを読み込みました。

口コミの感情分析を実行中...

1. 店内は落ち着いた雰囲気で、コーヒーもとても美味しかったです。... → POSITIVE (61.54%)
2. 混雑していて席がなかなか空きませんでした。あまりおすすめでき... → NEGATIVE (61.99%)
3. 混雑していて席がなかなか空きませんでした。次は別のお店を利用... → NEGATIVE (56.55%)
4. スタッフの対応が丁寧で気持ちよく過ごせました。友人にもおすす... → POSITIVE (56.13%)
5. アクセスが良く、また来たいと思えるカフェでした。良い時間を過... → POSITIVE (60.52%)
6. スイーツが美味しく、特にケーキが印象的でした。リピートしたい... → POSITIVE (58.88%)
7. 静かで作業にも集中できる良い環境でした。また利用したいです。... → POSITIVE (55.54%)
8. スイーツが美味しく、特にケーキが印象的でした。良い時間を過ご... → POSITIVE (62.67%)
9. アクセスが良く、また来たいと思えるカフェでした。リピートした... → POSITIVE (56.53%)
10. 店内は落ち着いた雰囲気で、コーヒーもとても美味しかったです。... → POSITIVE (59.25%)
11. スタッフの対応が丁寧で気持ちよく過ごせました。友人にもおすす... → POSITIVE (56.13%)
12. 店内が騒がしく、あまり落ち着けませんでした。次は別のお店を利... → NEGATIVE (56.87%)
13. アクセスが良く、また来たいと思えるカフェでした。また利用した... → POSITIVE (56.25%)
14. 店内は落ち着いた雰囲気で、コーヒーもとても美味しかったです。... → POSITIVE (58.98%)
15. 価格の割に満足度はあまり高くありませんでした。あまりおすすめ... → NEUTRAL (50.00%)
16. スタッフの対応が少し冷たく感じました。再訪は検討します。... → NEUTRAL (50.00%)
17. スタッフの対応が少し冷たく感じました。少し残念でした。... → NEUTRAL (50.0

,ID,評価(1-5),口コミ,感情,感情スコア
0,1,4,店内は落ち着いた雰囲気で、コーヒーもとても美味しかったです。良い時間を過ごせました。,POSITIVE,0.615366
1,2,2,混雑していて席がなかなか空きませんでした。あまりおすすめできません。,NEGATIVE,0.619908
2,3,1,混雑していて席がなかなか空きませんでした。次は別のお店を利用するかもしれません。,NEGATIVE,0.565491
3,4,4,スタッフの対応が丁寧で気持ちよく過ごせました。友人にもおすすめしたいです。,POSITIVE,0.561349
4,5,5,アクセスが良く、また来たいと思えるカフェでした。良い時間を過ごせました。,POSITIVE,0.605243


## 感情分析結果の解析

### Positive / Negative の件数と割合

In [ ]:
# 感情ごとの件数
sentiment_count = df["感情"].value_counts()

# 感情ごとの割合
sentiment_ratio = df["感情"].value_counts(normalize=True) * 100

# 表にまとめる
sentiment_summary = pd.DataFrame({
    "件数": sentiment_count,
    "割合(%)": sentiment_ratio.round(1)
})

display(sentiment_summary)


# 棒グラフ作成
import matplotlib.pyplot as plt
import matplotlib_fontja

sentiment_count.plot(
    kind="bar",
    figsize=(6, 4)
)

plt.title("口コミの感情分析結果")
plt.xlabel("感情")
plt.ylabel("口コミ数")
plt.xticks(rotation=0)
plt.show()

### 星評価と感情分析結果を比較

In [ ]:
# 評価（1～5）ごとに、各感情が占める割合を計算
cross_ratio = pd.crosstab(
    df["評価(1-5)"],
    df["感情"],
    normalize="index"
) * 100

# 集計結果を小数第1位まで表示
display(cross_ratio.round(1))

# グラフを作成
fig, ax = plt.subplots(
    figsize=(8, 5),
    layout="constrained"
)

# 評価ごとの感情割合を積み上げ棒グラフで表示
cross_ratio.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    legend=False
)

# グラフのタイトルと軸ラベルを設定
ax.set_title("評価ごとの口コミ感情の割合")
ax.set_xlabel("評価（1～5）")
ax.set_ylabel("割合（%）")

# 横軸のラベルを横向きに表示
ax.tick_params(
    axis="x",
    rotation=0
)

# レジェンドをグラフの右外側に表示
fig.legend(
    title="感情",
    loc="outside right upper"
)

# グラフを表示
plt.show()